# Phase 2 — Load Cookie Cats Dataset into SQL Server

**Project:** Cookie Cats A/B Test Analysis (Tactile Entertainment)
**Phase:** 2 — Data Understanding & EDA
**Notebook goal:** Load the raw `cookie_cats.csv` dataset, fingerprint it for reproducibility (MD5), push it into Microsoft SQL Server (`dbo.raw_ab_test`), and verify that the loaded table matches the source DataFrame exactly.

## Cell 1 — Imports & load CSV with explicit dtypes

We declare dtypes explicitly so the schema is deterministic (no silent type inference across mirrors of the dataset).

In [1]:
import pandas as pd
import numpy as np
import hashlib
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()

DATA_PATH = Path("../data/cookie_cats.csv")

df = pd.read_csv(
    DATA_PATH,
    dtype={
        "userid": "int64",
        "version": "string",
        "sum_gamerounds": "int64",
        "retention_1": "bool",
        "retention_7": "bool",
    },
)

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Shape: (90189, 5)
Columns: ['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7']


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


## Cell 2 — MD5 fingerprint for reproducibility

The MD5 hash pins the exact file used, so future runs (or a different Kaggle mirror) can be detected. Record this hash in the project log.

In [2]:
def file_md5(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

file_hash = file_md5(DATA_PATH)
print(f"MD5: {file_hash}")
print(f"Rows: {len(df)}")

MD5: 99b48ea3d4a552fa6b27aac60a8cfddf
Rows: 90189


## Cell 3 — Connect to SQL Server via SQLAlchemy

Connection string is assembled from `.env`. Windows Authentication (`Trusted_Connection=yes`) is the default; SQL Auth branch reads `DB_USER`/`DB_PASSWORD` only if `DB_TRUSTED_CONNECTION=no`. No credentials are hard-coded.

In [3]:
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

server = os.getenv("DB_SERVER")
database = os.getenv("DB_NAME")
driver = os.getenv("DB_DRIVER")
trusted = os.getenv("DB_TRUSTED_CONNECTION")

if trusted.lower() == "yes":
    conn_str = (
        f"DRIVER={{{driver}}};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"Trusted_Connection=yes;"
    )
else:
    user = os.getenv("DB_USER")
    password = os.getenv("DB_PASSWORD")
    conn_str = (
        f"DRIVER={{{driver}}};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"UID={user};PWD={password};"
    )

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={quote_plus(conn_str)}",
    fast_executemany=True,
)

with engine.connect() as conn:
    result = conn.execute(text("SELECT @@VERSION")).fetchone()
    print(result[0][:80])

Microsoft SQL Server 2022 (RTM) - 16.0.1000.6 (X64) 
	Oct  8 2022 05:58:25 
	Cop


## Cell 4 — Load DataFrame into `dbo.raw_ab_test`

SQL Server has no native boolean, so `retention_1` / `retention_7` are cast to `int` (stored as `BIT`). `if_exists="replace"` makes the load idempotent.

In [4]:
df_sql = df.copy()
df_sql["retention_1"] = df_sql["retention_1"].astype(int)
df_sql["retention_7"] = df_sql["retention_7"].astype(int)

df_sql.to_sql(
    name="raw_ab_test",
    con=engine,
    schema="dbo",
    if_exists="replace",
    index=False,
    chunksize=10_000,
)

print("Loaded to dbo.raw_ab_test")

Loaded to dbo.raw_ab_test


## Cell 5 — Verify: SQL Server matches the DataFrame

Assertions fail loudly on any row-count or unique-user mismatch. **If `total_rows` ≠ 90,189, stop** — the source file may be a different mirror or corrupt.

In [5]:
verify_query = text("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT userid) AS unique_users,
    COUNT(DISTINCT version) AS n_versions,
    MIN(sum_gamerounds) AS min_rounds,
    MAX(sum_gamerounds) AS max_rounds
FROM dbo.raw_ab_test;
""")

with engine.connect() as conn:
    verify_df = pd.read_sql(verify_query, conn)

print(verify_df)

assert verify_df["total_rows"].iloc[0] == len(df), "Row count mismatch!"
assert verify_df["unique_users"].iloc[0] == df["userid"].nunique(), "User count mismatch!"
print("\u2713 SQL Server data matches DataFrame")

   total_rows  unique_users  n_versions  min_rounds  max_rounds
0       90189         90189           2           0       49854
✓ SQL Server data matches DataFrame
